[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/getting-started/datasets.ipynb)

# The Datasets: Five Real-World Case Studies

Every example in this course runs on **real data**, not a cleaned-up toy. Real data is late, duplicated, mis-typed and sometimes plainly wrong, and learning to handle that is half of machine learning in practice.

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Load any course dataset with one line of code.</li>
<li>Describe what each dataset is, where it comes from, and what question we ask of it.</li>
<li>Name at least two concrete quality problems in each dataset <em>before</em> modelling it.</li>
</ul>
</div>

**Prerequisites:** Python and pandas basics. **Time:** about 25 minutes (first run downloads ~150 MB).

## Setup

The cell below installs the course helper package `mlp_data` if you do not have it (this is what happens on Google Colab). Datasets are downloaded from their original publishers the first time you use them, then cached on your machine.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import pandas as pd
from mlp_data import load_retail, load_movielens, load_taxi, load_airbnb, load_credit

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)

<div class="admonition note">
<p class="admonition-title">Where does the data go?</p>
<p>Files are cached in <code>~/.cache/mlp_course</code> (override with the <code>MLP_DATA_DIR</code> environment variable). Nothing is stored in the course repository. If a download fails, the error message says why; a changed file triggers a checksum warning rather than silently giving you different numbers.</p>
</div>

---

## 1. Online Retail II: the main thread

**What it is.** Every invoice line of a UK online gift-ware retailer for two years (Dec 2009 to Dec 2011). Many customers are wholesalers. *Source: UCI Machine Learning Repository, CC BY 4.0.*

**Why we use it.** One dataset supports four different problems across the course: predicting which customers will return (classification), grouping customers (clustering), forecasting demand (time series), and suggesting products (recommenders).

The first load downloads ~46 MB and converts an Excel file, which takes about 35 seconds. After that it is nearly instant.

In [2]:
retail = load_retail()
print(retail.shape)
retail.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Before modelling anything, **audit it**. Every number below is computed from the data, not quoted from a description:

In [3]:
audit = pd.Series({
    "rows": len(retail),
    "first invoice date": retail["InvoiceDate"].min().date(),
    "last invoice date": retail["InvoiceDate"].max().date(),
    "rows with no Customer ID (%)": round(retail["Customer ID"].isna().mean() * 100, 1),
    "cancellation invoices (start with 'C')": int(retail["Invoice"].str.startswith("C").sum()),
    "rows with Quantity <= 0": int((retail["Quantity"] <= 0).sum()),
    "rows with Price <= 0": int((retail["Price"] <= 0).sum()),
    "exact duplicate rows": int(retail.duplicated().sum()),
}, name="retail audit")
audit.to_frame()

,retail audit
rows,1067371
first invoice date,2009-12-01
last invoice date,2011-12-09
rows with no Customer ID (%),22.8
cancellation invoices (start with 'C'),19494
rows with Quantity <= 0,22950
rows with Price <= 0,6207
exact duplicate rows,34335


<div class="admonition warning">
<p class="admonition-title">What this means</p>
<p>Roughly one row in five has no customer, so any per-customer analysis silently ignores those rows. Negative quantities are returns and cancellations, not errors, but a naive "total sales" sum will be wrong unless you decide how to treat them. We deal with all of this properly in the lifecycle chapters.</p>
</div>

Some `StockCode` values are not products at all:

In [4]:
non_product = retail[retail["StockCode"].str.fullmatch(r"[A-Za-z]+", na=False)]
non_product["StockCode"].value_counts().head(8)

StockCode
POST         2122
DOT          1446
M            1421
D             177
S             104
ADJUST         67
AMAZONFEE      43
DCGSSGIRL      25
Name: count, dtype: int64[pyarrow]

---

## 2. MovieLens: recommender systems

**What it is.** Star ratings (0.5 to 5) that real users gave to movies on the MovieLens service. We use the "small" version. *Source: GroupLens Research, University of Minnesota. Free for non-commercial and educational use; see the attribution section at the end.*

**Why we use it.** Explicit ratings are the cleanest way to meet the core recommender ideas (collaborative filtering, matrix factorisation, ranking metrics) before we move to the messier "implicit" purchases in the retail data.

In [5]:
ml = load_movielens()
print({name: df.shape for name, df in ml._asdict().items()})
ml.ratings.merge(ml.movies, on="movieId").head()

{'ratings': (100836, 4), 'movies': (9742, 3), 'links': (9742, 3), 'tags': (3683, 4)}


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [6]:
n_users, n_items = ml.ratings["userId"].nunique(), ml.ratings["movieId"].nunique()
sparsity = 1 - len(ml.ratings) / (n_users * n_items)
per_user = ml.ratings.groupby("userId").size()
pd.Series({
    "users": n_users,
    "movies rated": n_items,
    "ratings": len(ml.ratings),
    "sparsity (share of user-movie pairs with no rating)": f"{sparsity:.1%}",
    "ratings per user: min / median / max": f"{per_user.min()} / {per_user.median():.0f} / {per_user.max()}",
}).to_frame("movielens")

,movielens
users,610
movies rated,9724
ratings,100836
sparsity (share of user-movie pairs with no rating),98.3%
ratings per user: min / median / max,20 / 70 / 2698


Almost every cell of the user-by-movie table is empty. Recommending is the art of filling it in, and that sparsity is why it is harder than it looks.

---

## 3. NYC Taxi Trips: time series

**What it is.** Every yellow-cab trip in New York City, published monthly by the city's Taxi & Limousine Commission (TLC). *Source: NYC TLC trip record data.*

**Why we use it.** Millions of timestamped events with strong daily and weekly rhythms, holidays, and a pandemic-sized disruption: the real conditions a forecasting model has to survive. One month is about 3 million rows (a 48 MB download).

In [7]:
taxi = load_taxi("2023-01")
print(taxi.shape)
taxi[["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
      "trip_distance", "PULocationID", "fare_amount", "total_amount"]].head()

(3066766, 19)


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,fare_amount,total_amount
0,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,161,9.3,14.30
1,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,43,7.9,16.90
2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,48,14.9,34.90
3,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,138,12.1,20.85
4,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,107,11.4,19.68


In [8]:
pickup = taxi["tpep_pickup_datetime"]
in_month = pickup.dt.to_period("M") == "2023-01"
pd.Series({
    "trips": len(taxi),
    "earliest pickup timestamp": pickup.min(),
    "pickups that fall OUTSIDE January 2023": int((~in_month).sum()),
    "trips with a negative fare": int((taxi["fare_amount"] < 0).sum()),
    "trips with zero distance": int((taxi["trip_distance"] == 0).sum()),
    "trips with missing passenger_count": int(taxi["passenger_count"].isna().sum()),
}).to_frame("taxi audit")

,taxi audit
trips,3066766
earliest pickup timestamp,2008-12-31 23:01:42
pickups that fall OUTSIDE January 2023,48
trips with a negative fare,25049
trips with zero distance,45862
trips with missing passenger_count,71743


A file named *January 2023* contains trips from 2008. These are almost certainly meter or clock errors, and they would wreck any time axis if you did not look.

---

## 4. Inside Airbnb (London): messy tabular regression

**What it is.** A snapshot of every Airbnb listing in London scraped by the Inside Airbnb project, with prices, locations, host details, review scores and free text. *Source: Inside Airbnb, CC BY 4.0. Snapshot date 2026-09-17.*

**Why we use it.** 90 columns of mixed types, heavy missingness and text: ideal for feature engineering and for asking "what drives price?" (~48 MB download).

In [9]:
airbnb = load_airbnb()
print(airbnb.shape)
airbnb[["id", "neighbourhood_cleansed", "room_type", "accommodates",
        "bedrooms", "price", "number_of_reviews", "review_scores_rating"]].head()

(91954, 90)


,id,neighbourhood_cleansed,room_type,accommodates,bedrooms,price,number_of_reviews,review_scores_rating
0,11551,Lambeth,Entire home/apt,5,1.0,$267.50,197,4.55
1,13913,Islington,Private room,1,1.0,$127.00,57,4.86
2,15400,Kensington and Chelsea,Entire home/apt,2,1.0,$137.50,98,4.81
3,24328,Wandsworth,Entire home/apt,2,1.0,$228.00,97,4.90
4,25081,Haringey,Entire home/apt,4,2.0,NaN,0,NaN


In [10]:
pd.Series({
    "price column dtype": str(airbnb["price"].dtype),
    "example price values": ", ".join(airbnb["price"].dropna().head(3)),
    "price missing (%)": round(airbnb["price"].isna().mean() * 100, 1),
    "review_scores_rating missing (%)": round(airbnb["review_scores_rating"].isna().mean() * 100, 1),
    "columns": airbnb.shape[1],
}).to_frame("airbnb audit")

,airbnb audit
price column dtype,str
example price values,"$267.50, $127.00, $137.50"
price missing (%),34.9
review_scores_rating missing (%),22.5
columns,90


The target we would want to predict, **price, is stored as text and a third of it is missing.** You cannot even call `.mean()` on it yet.

<div class="admonition tip">
<p class="admonition-title">Link rot</p>
<p>Inside Airbnb keeps only about a year of snapshots, so this exact file will eventually disappear from their site. We pin one snapshot and verify its checksum. If you ever see a 404 here, the course needs a mirror: please open an issue.</p>
</div>

---

## 5. Credit Card Default (Taiwan, 2005): imbalanced classification

**What it is.** 30,000 credit-card customers in Taiwan with their repayment history, and whether they defaulted the following month. *Source: UCI Machine Learning Repository, CC BY 4.0.*

**Why we use it.** Only about one customer in five defaults, so plain accuracy is misleading. It also records `SEX` and `AGE`, which lets us discuss fairness. It is **2005 data from one country**: use it to learn methods, not to draw conclusions about lending today.

In [11]:
credit = load_credit()
print(credit.shape)
credit.iloc[:, [0, 1, 2, 3, 4, 5, 6, -1]].head()

(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,default payment next month
0,1,20000,2,2,1,24,2,1
1,2,120000,2,2,2,26,-1,1
2,3,90000,2,2,2,34,0,0
3,4,50000,2,2,1,37,0,0
4,5,50000,1,2,1,57,-1,0


In [12]:
target = "default payment next month"
print(f"default rate: {credit[target].mean():.1%}  ->  a model that always says 'no default' is {1 - credit[target].mean():.1%} accurate")
print("EDUCATION codes present:", [int(v) for v in sorted(credit["EDUCATION"].unique())])
print("MARRIAGE  codes present:", [int(v) for v in sorted(credit["MARRIAGE"].unique())])

default rate: 22.1%  ->  a model that always says 'no default' is 77.9% accurate
EDUCATION codes present: [0, 1, 2, 3, 4, 5, 6]
MARRIAGE  codes present: [0, 1, 2, 3]


The dataset documentation defines education codes 1 to 4 only, yet codes **0, 5 and 6 appear**. Undocumented values like these are everywhere in real data; the right response is to investigate and decide, not to guess.

---

## Check yourself

<details class="question">
<summary>1. In the retail data, how many distinct customers are there (ignoring rows with no customer)? Try it first, then open the answer.</summary>

```python
retail["Customer ID"].nunique()
```

`nunique()` skips missing values by default, so the null customer IDs do not inflate the count.

</details>

<details class="question">
<summary>2. In the taxi data, which pickups fall outside January 2023, and what is the earliest year? Try it first.</summary>

```python
outside = taxi[~in_month]
outside["tpep_pickup_datetime"].dt.year.value_counts()
```

`in_month` was built in the audit cell above. Counting by year shows how many stray rows come from each year.

</details>

<details class="question">
<summary>3. Why would "accuracy" be a poor way to judge a credit-default model on this data?</summary>

About 78% of customers do **not** default, so a model that always predicts "no default" is about 78% accurate while catching zero defaulters. We will use precision, recall and PR/ROC curves instead.

</details>

## Attribution and licences

| Dataset | Credit | Terms |
|---|---|---|
| Online Retail II | Chen, D. (2019), UCI Machine Learning Repository, doi:10.24432/C5CG6D | CC BY 4.0 |
| MovieLens small | Harper & Konstan (2015), *The MovieLens Datasets: History and Context*, ACM TiiS 5(4) | Non-commercial use only; may be redistributed under the same terms |
| NYC Yellow Taxi | NYC Taxi & Limousine Commission | Downloaded on demand, never redistributed by this course |
| Inside Airbnb | Inside Airbnb (insideairbnb.com) | CC BY 4.0 |
| Credit default | Yeh, I-C. (2016), UCI Machine Learning Repository, doi:10.24432/C55S3H | CC BY 4.0 |

Full provenance, checksums and open issues for each dataset are recorded in `data/README.md` in the course repository.

**Next:** *How to use this site* and the *learning path* explain the order to read things in.